In [1]:
# ─────────────────────────────────────────────────────
# Cell 1: Install
# ─────────────────────────────────────────────────────
!pip install easyocr datasets jiwer -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 15.2 MB/s eta 0:00:00


In [2]:
# ─────────────────────────────────────────────────────
# Cell 2: Import
# ─────────────────────────────────────────────────────
import re
import json
import os
import easyocr
from datasets import load_dataset
from jiwer import cer, wer

In [3]:
# ─────────────────────────────────────────────────────
# Cell 3: Load dataset
# ─────────────────────────────────────────────────────
baseline_dataset = load_dataset("Teklia/IAM-line", split="test[:500]")
print(f"Dataset loaded: {len(baseline_dataset)} samples")

README.md: 0.00B [00:00, ?B/s]

data/train.parquet:   0%|          | 0.00/167M [00:00<?, ?B/s]

data/validation.parquet:   0%|          | 0.00/24.7M [00:00<?, ?B/s]

data/test.parquet:   0%|          | 0.00/73.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6482 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/976 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2915 [00:00<?, ? examples/s]

Dataset loaded: 500 samples


In [4]:
# ─────────────────────────────────────────────────────
# Cell 4: Cek sample pertama (sama seperti DeepSeek-mu)
# ─────────────────────────────────────────────────────
sample = baseline_dataset[0]

sample["image"].save("temp.jpg")
sample["image"]

# ─────────────────────────────────────────────────────
# Cell 5: Cek ground truth
# ─────────────────────────────────────────────────────
ground_truth = sample["text"]
print("GT:", ground_truth)


GT: assuredness " Bella Bella Marie " ( Parlophone ) , a lively song that changes tempo mid-way .


In [5]:
# ─────────────────────────────────────────────────────
# Cell 6: Normalisasi teks
# ─────────────────────────────────────────────────────
def normalize_text(text):
    text = text.strip()
    text = text.replace("\n", " ")       # newline → space
    text = re.sub(r" +", " ", text)      # multi-space → single space
    text = text.lower()                  # lowercase (case-insensitive)
    return text

In [6]:
# ─────────────────────────────────────────────────────
# Cell 7: Load EasyOCR model (run once)
# ─────────────────────────────────────────────────────
reader = easyocr.Reader(['en'], gpu=True)
print("EasyOCR model loaded ✓")

EasyOCR model loaded ✓


In [7]:
# ─────────────────────────────────────────────────────
# Cell 8: Evaluate EasyOCR baseline
# (mengikuti struktur loop DeepSeek-mu)
# ─────────────────────────────────────────────────────
os.makedirs("./results", exist_ok=True)

baseline_total_cer = 0
baseline_total_wer = 0

for i, sample in enumerate(baseline_dataset):

    # Simpan image sementara
    sample["image"].save("temp.jpg")
    gt = normalize_text(sample["text"])

    # Jalankan OCR
    results = reader.readtext("temp.jpg", detail=0)  # detail=0 = teks saja
    pred = normalize_text(" ".join(results))

    # Fallback jika kosong
    if not gt.strip():   gt   = " "
    if not pred.strip(): pred = " "

    # Hitung metric
    baseline_total_cer += cer(gt, pred)
    baseline_total_wer += wer(gt, pred)

    print(f"Done {i+1}/{len(baseline_dataset)}")

Done 1/500
Done 2/500
Done 3/500
Done 4/500
Done 5/500
Done 6/500
Done 7/500
Done 8/500
Done 9/500
Done 10/500
Done 11/500
Done 12/500
Done 13/500
Done 14/500
Done 15/500
Done 16/500
Done 17/500
Done 18/500
Done 19/500
Done 20/500
Done 21/500
Done 22/500
Done 23/500
Done 24/500
Done 25/500
Done 26/500
Done 27/500
Done 28/500
Done 29/500
Done 30/500
Done 31/500
Done 32/500
Done 33/500
Done 34/500
Done 35/500
Done 36/500
Done 37/500
Done 38/500
Done 39/500
Done 40/500
Done 41/500
Done 42/500
Done 43/500
Done 44/500
Done 45/500
Done 46/500
Done 47/500
Done 48/500
Done 49/500
Done 50/500
Done 51/500
Done 52/500
Done 53/500
Done 54/500
Done 55/500
Done 56/500
Done 57/500
Done 58/500
Done 59/500
Done 60/500
Done 61/500
Done 62/500
Done 63/500
Done 64/500
Done 65/500
Done 66/500
Done 67/500
Done 68/500
Done 69/500
Done 70/500
Done 71/500
Done 72/500
Done 73/500
Done 74/500
Done 75/500
Done 76/500
Done 77/500
Done 78/500
Done 79/500
Done 80/500
Done 81/500
Done 82/500
Done 83/500
Done 84/500
D

In [8]:
# Cleanup
if os.path.exists("temp.jpg"):
    os.remove("temp.jpg")

# ─────────────────────────────────────────────────────
# Cell 9: Hasil akhir
# ─────────────────────────────────────────────────────
baseline_avg_cer = baseline_total_cer / len(baseline_dataset)
baseline_avg_wer = baseline_total_wer / len(baseline_dataset)

print("=== EasyOCR BASELINE RESULT ===")
print(f"  Model   : EasyOCR (en)")
print(f"  Dataset : Teklia/IAM-line")
print(f"  Samples : {len(baseline_dataset)}")
print(f"  CER     : {baseline_avg_cer:.4f}")
print(f"  WER     : {baseline_avg_wer:.4f}")


results = {
    "model":       "EasyOCR",
    "dataset":     "Teklia/IAM-line",
    "split":       "test[:1000]",
    "num_samples": len(baseline_dataset),
    "avg_cer":     baseline_avg_cer,
    "avg_wer":     baseline_avg_wer,
}

with open("./results/easyocr_baseline_results.json", "w") as f:
    json.dump(results, f, indent=4)

print(f"\n[Saved] Hasil disimpan ke ./results/easyocr_baseline_results.json")

=== EasyOCR BASELINE RESULT ===
  Model   : EasyOCR (en)
  Dataset : Teklia/IAM-line
  Samples : 500
  CER     : 0.5858
  WER     : 1.0236

[Saved] Hasil disimpan ke ./results/easyocr_baseline_results.json
